# Optimize SCALP-lite Parameters

Download datasets first with notebook 00. This notebook selects one dataset, optimizes preprocessing PCA and graph parameters, then saves the best settings for downstream notebooks.


In [1]:
import sys
import subprocess
import warnings

warnings.filterwarnings(
    "ignore",
    message=r"\s*Found Intel OpenMP .* LLVM OpenMP .*",
    category=RuntimeWarning,
    module=r"threadpoolctl",
)


def ensure_bo_dependencies():
    try:
        import botorch  # noqa: F401
        import gpytorch  # noqa: F401
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", "..[bo]"])


ensure_bo_dependencies()


In [2]:
import numpy as np

from scalp_lite.metrics import score_embedding
from scalp_lite.notebook_utils import dataset_config, make_estimator, save_optimized_graph_params
from scalp_lite.optimization import latent_bayesopt


selected_dataset = "pancreas"
dataset = dataset_config(selected_dataset)

# Keep objective evaluations manageable. Increase after validating the search setup.
fixed_preprocess_params = {"max_cells": 2000}
random_state = 0
raw_estimator = make_estimator(dataset, n_components=100, random_state=random_state)
raw_adata = raw_estimator.to_data(dataset["input_path"])
raw_adata


AnnData object with n_obs × n_vars = 15681 × 1000
    obs: 'batch', 'study', 'cell_type', 'size_factors'

In [3]:
base_preprocess_params = {
    "n_top_genes": 2000,
}

base_estimator_params = {
    "n_components": 100,
}

base_graph_params = {
    "n_neighbors": 15,
    "intra_fraction": 0.5,
    "n_inter_edges": 2,
    "metric": "euclidean",
    "assignment_quantile": 0.8,
    "hubness_correction": "csls",
    "hubness_k": 10,
    "rank_correction": True,
    "edge_weighting": "binary",
    "mutual_neighbors": False,
    "neighbor_mode": "distance",
    "symmetrize": True,
}

preprocess_search_space = {
    "n_top_genes": {"type": "int", "bounds": [500, 3000]},
}

estimator_search_space = {
    "n_components": {"type": "int", "bounds": [20, 150]},
}

graph_search_space = {
    "n_neighbors": {"type": "int", "bounds": [5, 40]},
    "intra_fraction": {"type": "float", "bounds": [0.2, 0.9]},
    "n_inter_edges": {"type": "int", "bounds": [1, 8]},
    "assignment_quantile": {"type": "float", "bounds": [0.05, 1.0]},
    "hubness_k": {"type": "int", "bounds": [3, 30]},
    "rank_correction": {"type": "categorical", "values": [False, True]},
    "edge_weighting": {"type": "categorical", "values": ["binary", "distance"]},
    "mutual_neighbors": {"type": "categorical", "values": [False, True]},
}

search_space = {
    **preprocess_search_space,
    **estimator_search_space,
    **graph_search_space,
}

base_preprocess_params, base_estimator_params, base_graph_params, search_space


({'n_top_genes': 2000},
 {'n_components': 100},
 {'n_neighbors': 15,
  'intra_fraction': 0.5,
  'n_inter_edges': 2,
  'metric': 'euclidean',
  'assignment_quantile': 0.8,
  'hubness_correction': 'csls',
  'hubness_k': 10,
  'rank_correction': True,
  'edge_weighting': 'binary',
  'mutual_neighbors': False,
  'neighbor_mode': 'distance',
  'symmetrize': True},
 {'n_top_genes': {'type': 'int', 'bounds': [500, 3000]},
  'n_components': {'type': 'int', 'bounds': [20, 150]},
  'n_neighbors': {'type': 'int', 'bounds': [5, 40]},
  'intra_fraction': {'type': 'float', 'bounds': [0.2, 0.9]},
  'n_inter_edges': {'type': 'int', 'bounds': [1, 8]},
  'assignment_quantile': {'type': 'float', 'bounds': [0.05, 1.0]},
  'hubness_k': {'type': 'int', 'bounds': [3, 30]},
  'rank_correction': {'type': 'categorical', 'values': [False, True]},
  'edge_weighting': {'type': 'categorical', 'values': ['binary', 'distance']},
  'mutual_neighbors': {'type': 'categorical', 'values': [False, True]}})

In [4]:
preprocess_param_names = set(preprocess_search_space)
estimator_param_names = set(estimator_search_space)
graph_param_names = set(graph_search_space)


def split_trial_params(params):
    preprocess_params = {
        **base_preprocess_params,
        **fixed_preprocess_params,
        **{key: params[key] for key in preprocess_param_names if key in params},
    }
    estimator_params = {
        **base_estimator_params,
        **{key: params[key] for key in estimator_param_names if key in params},
    }
    graph_params = {
        **base_graph_params,
        **{key: params[key] for key in graph_param_names if key in params},
    }
    return preprocess_params, estimator_params, graph_params


def metric_value(scores, key, default=0.0):
    value = float(scores.get(key, default))
    return default if not np.isfinite(value) else value


def scalp_objective(params):
    preprocess_params, estimator_params, graph_params = split_trial_params(params)
    trial_estimator = make_estimator(dataset, random_state=random_state, **estimator_params)
    try:
        trial = trial_estimator.preprocess(raw_adata, **preprocess_params)
        trial.obsm["X_scalp"] = trial_estimator.embed(
            trial,
            **graph_params,
            embedding_random_state=random_state,
            n_epochs=100,
        )
        scores = score_embedding(
            trial,
            embedding_key="X_scalp",
            batch_key=dataset["batch_key"],
            label_key=dataset["label_key"] if dataset["label_key"] in trial.obs else None,
        ).iloc[0]
    except Exception as exc:
        print(
            "failed params="
            f"preprocess={preprocess_params}, estimator={estimator_params}, graph={graph_params}: {exc}"
        )
        return -1e9

    label_agreement = metric_value(scores, "knn_label_agreement")
    batch_mixing = metric_value(scores, "batch_mixing")
    graph_density = metric_value(scores, "graph_density")
    # Maximize biological coherence and batch mixing; tune weights for your target use case.
    return float(label_agreement + 0.25 * batch_mixing - 0.05 * graph_density)


In [ ]:
# First pass: broad PCA latent BO for a cheap sanity-check search.
bo_result = latent_bayesopt(
    scalp_objective,
    search_space,
    n_initial=12,
    latent_dim=3,
    n_iterations=12,
    embedding_model="pca",
    acquisition="ei",
    batch_size=1,
    random_state=random_state,
    invalid_score=-1e9,
)

bo_result["best_params"], bo_result["best_score"]


In [ ]:
bo_result["history"].sort_values("score", ascending=False).head(10)


In [ ]:
def compact_bounds(best, name, low, high, radius, *, integer=False):
    center = best[name]
    new_low = max(low, center - radius)
    new_high = min(high, center + radius)
    if integer:
        new_low = int(round(new_low))
        new_high = int(round(new_high))
        if new_low == new_high:
            new_low = max(low, new_low - 1)
            new_high = min(high, new_high + 1)
    return [new_low, new_high]


best_pca_params = bo_result["best_params"]

# Second-stage search ranges, centered on the good PCA solution but still wide enough to move.
compact_search_space = {
    "n_top_genes": {
        "type": "int",
        "bounds": compact_bounds(best_pca_params, "n_top_genes", 500, 3000, 500, integer=True),
    },
    "n_components": {
        "type": "int",
        "bounds": compact_bounds(best_pca_params, "n_components", 20, 150, 30, integer=True),
    },
    "n_neighbors": {
        "type": "int",
        "bounds": compact_bounds(best_pca_params, "n_neighbors", 5, 40, 8, integer=True),
    },
    "intra_fraction": {
        "type": "float",
        "bounds": compact_bounds(best_pca_params, "intra_fraction", 0.2, 0.9, 0.12),
    },
    "n_inter_edges": {
        "type": "int",
        "bounds": compact_bounds(best_pca_params, "n_inter_edges", 1, 8, 2, integer=True),
    },
    "assignment_quantile": {
        "type": "float",
        "bounds": compact_bounds(best_pca_params, "assignment_quantile", 0.05, 1.0, 0.15),
    },
    "hubness_k": {
        "type": "int",
        "bounds": compact_bounds(best_pca_params, "hubness_k", 3, 30, 6, integer=True),
    },
    # Keep the categorical choices that worked in the PCA stage fixed for the refinement.
    "rank_correction": {"type": "categorical", "values": [best_pca_params["rank_correction"]]},
    "edge_weighting": {"type": "categorical", "values": [best_pca_params["edge_weighting"]]},
    "mutual_neighbors": {"type": "categorical", "values": [best_pca_params["mutual_neighbors"]]},
}

compact_search_space


In [ ]:
# Second pass: nonlinear GPLVM latent BO over the compact space around the PCA solution.
gplvm_result = latent_bayesopt(
    scalp_objective,
    compact_search_space,
    n_initial=12,
    latent_dim=3,
    n_iterations=12,
    embedding_model="gplvm",
    acquisition="ei",
    batch_size=1,
    random_state=random_state + 1,
    invalid_score=-1e9,
)

gplvm_result["best_params"], gplvm_result["best_score"]


In [ ]:
gplvm_result["history"].sort_values("score", ascending=False).head(10)


In [ ]:
optimization_results = {"pca": bo_result, "gplvm": gplvm_result}
best_model_name, best_result = max(
    optimization_results.items(),
    key=lambda item: item[1]["best_score"],
)

optimized_preprocess_params, optimized_estimator_params, optimized_graph_params = split_trial_params(best_result["best_params"])
params_path = save_optimized_graph_params(
    selected_dataset,
    optimized_graph_params,
    preprocess_params=optimized_preprocess_params,
    estimator_params=optimized_estimator_params,
    metadata={
        "best_model": best_model_name,
        "best_score": best_result["best_score"],
        "pca_best_score": bo_result["best_score"],
        "gplvm_best_score": gplvm_result["best_score"],
        "fixed_preprocess_params": fixed_preprocess_params,
        "random_state": random_state,
    },
)

params_path, optimized_preprocess_params, optimized_estimator_params, optimized_graph_params
